In [ ]:
import requests

# Use the export domain for automated downloads
remote_pdf_url = "https://export.arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "ch02-downloaded.pdf"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(remote_pdf_url, headers=headers)

if response.status_code == 200:
    with open(pdf_filename, "wb") as pdf_file:
        pdf_file.write(response.content)
    print("Download successful!")
else:
    print(f"Failed to download pdf. Status code: {response.status_code}")

In [ ]:
import pdfplumber

text = ""

with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

print(text[0:20])


In [ ]:
from utils import chunk_text

chunks = chunk_text(text, 500, 40)
print(len(chunks))
print(chunks[0])

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# load env variables
load_dotenv()

OPEN_AI_KEY = os.getenv("OPENAI_API_KEY")
open_ai_client = OpenAI(api_key=OPEN_AI_KEY)

In [ ]:
def embed(texts):
    response = open_ai_client.embeddings.create(
        input=texts,
        model='text-embedding-3-small'
    )

    return list(map(lambda n: n.embedding, response.data))

embeddings = embed(chunks)

print(embeddings[0][0:3])
print(len(embeddings))
print(len(embeddings[0]))

In [21]:
load_dotenv(override=True)

neo4j_user = os.getenv("NEO4J_USER")
neo4j_pwd = os.getenv("NEO4J_PWD")
neo4j_uri = os.getenv("NEO4J_URI")

driver = GraphDatabase.driver(neo4j_uri, auth=(neo4j_user, neo4j_pwd))
driver.verify_connectivity(database="153d5309")
print("Connected!")


Connected!


In [ ]:
driver.execute_query("""CREATE VECTOR INDEX pdf IF NOT EXISTS
FOR (c:Chunk)
ON c.embedding""")

In [ ]:
cypher_query = '''
WITH $chunks as chunks, range(0, size($chunks)) AS index
UNWIND index AS i
WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
MERGE (c:Chunk {index: i})
SET c.text = chunk, c.embedding = embedding
'''

driver.execute_query(cypher_query, chunks=chunks, embeddings=embeddings)

In [ ]:
records, _, _ = driver.execute_query("MATCH (c:Chunk) WHERE c.index = 0 RETURN c.embedding, c.text")

print(records[0]["c.text"][0:30])
print(records[0]["c.embedding"][0:3])

In [ ]:
question = "At what time was Einstein really interested in experimental works?"
question_embedding = embed([question])[0]

query = '''
CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node AS hits, score
RETURN hits.text AS text, score, hits.index AS index
'''
similar_records, _, _ = driver.execute_query(query, question_embedding=question_embedding, k=4)

for record in similar_records:
    print(record["text"])
    print(record["score"], record["index"])
    print("======")